# Chapter 1 — FODO Cells


## Goals

1. Build a **forward arc FODO cell** with the same geometry as the original tutorial.
2. Compute the **periodic linear optics** with `twiss`.
3. Translate the original Tao optimization setup into a Julia objective function.
4. Use **Optim.jl** with a numerically evaluated gradient to tune the two quadrupole strengths so that the phase advance per cell is **90° in both planes**.

The traditional workflow is:

> geometry → initial optics → target definition → variables → optimization → verification

That is exactly the structure we keep here.


In [ ]:
import Pkg;
Pkg.add("SciBmad")
Pkg.add("Beamlines")
Pkg.add("CairoMakie")
Pkg.add("LaTeXStrings")
Pkg.add("Optim")
Pkg.add("DifferentiationInterface")
Pkg.add("GTPSA")

In [46]:
using SciBmad
using Beamlines
using CairoMakie
using LaTeXStrings
using Optim
using DifferentiationInterface
import DifferentiationInterface as DI
using GTPSA

set_theme!(theme_latexfonts())


## 1.2 Example: forward arc FODO cell

The forward arc FODO cell has the form:

$$
\mathrm{QF} \to D1 \to B \to D2 \to \mathrm{QD} \to D1 \to B \to D2.
$$

The geometry is:

- quadrupole length: $0.5\ \mathrm{m}$
- drift lengths: $D1 = 0.609\ \mathrm{m}$ and $D2 = 1.241\ \mathrm{m}$
- bend chord length: $L_{\mathrm{chord}} = 6.86\ \mathrm{m}$
- bend angle per dipole: $\theta_B = \pi/132$

In the original Bmad lattice, the bend is entered as an **rbend**, so the input length is the **chord length**. Here we build an `SBend` directly, so we first convert the chord length to the corresponding arc length.

In [ ]:
# Initial quadrupole strengths from the original Chapter 1 example
kqf0 = 0.4
kqd0 = -0.4

# Geometry of the forward arc FODO cell
Lq = 0.5
Ld1 = 0.609
Ld2 = 1.241
Lchord = 6.86
theta_b = π / 132

# The original tutorial specifies an rbend by chord length.
# Since we use SBend directly here, convert chord length -> arc length.
Lb = Lchord * theta_b / (2sin(theta_b / 2))

qf = Quadrupole(L=Lq, Kn1=kqf0)
d1 = Drift(L=Ld1)
b  = SBend(L=Lb, angle=theta_b)
d2 = Drift(L=Ld2)
qd = Quadrupole(L=Lq, Kn1=kqd0)

fodo0 = Beamline(
    [qf, d1, b, d2, qd, d1, b, d2],
    species_ref=Species("electron"),
    E_ref=18e9,
)

## 1.3 Inspect the initial optics

We check that the initial choice $K_1(QF)=0.4$ and $K_1(QD)=-0.4$ does **not yet** give the desired $90^\circ$ phase advance.


In [ ]:
zero6  = [0, 0, 0, 0, 0, 0]
delta1 = [0, 0, 0, 0, 0, 1]

function tps_const(x)
    try
        return x[zero6]
    catch
        return x
    end
end

function tps_delta_coeff(x)
    try
        return x[delta1]
    catch
        return zero(tps_const(x))
    end
end

function build_fodo_cell(kqf, kqd)
    qf = Quadrupole(L=0.5, Kn1=kqf)
    d1 = Drift(L=0.609)
    b  = SBend(L=6.86 * (π / 132) / (2sin((π / 132) / 2)), angle=π / 132)
    d2 = Drift(L=1.241)
    qd = Quadrupole(L=0.5, Kn1=kqd)

    return Beamline(
        [qf, d1, b, d2, qd, d1, b, d2],
        species_ref=Species("electron"),
        E_ref=18e9,
    )
end

# SciBmad 0.1 returns the optics table directly, while newer versions
# return a Twiss object containing the table.
optics_table(tw) = hasproperty(tw, :table) ? tw.table : tw

function cell_tunes(k)
    t = optics_table(twiss(build_fodo_cell(k[1], k[2])))
    return t.phi_1[end], t.phi_2[end]
end

function cell_phase_advances(k)
    Qx, Qy = cell_tunes(k)
    return 2π * Qx, 2π * Qy
end

function linear_dispersion(tw_table)
    return tw_table.eta_1, tw_table.eta_2
end

In [ ]:
tw0 = twiss(fodo0)
t0 = optics_table(tw0)

μx0, μy0 = cell_phase_advances([kqf0, kqd0])

println("Initial phase advances:")
println("  μx = ", μx0, " rad = ", μx0 * 180 / π, " deg")
println("  μy = ", μy0, " rad = ", μy0 * 180 / π, " deg")
println()
println("Target phase advance:")
println("  π/2 rad = 90 deg")

In [ ]:
function plot_linear_optics(tw; title_text="Initial linear optics")
    t = optics_table(tw)
    ηx, ηy = linear_dispersion(t)

    f = Figure(fontsize=20, size=(900, 700))

    ax1 = Axis(f[1, 1], xlabel="s [m]", ylabel="Beta [m]", title=title_text)
    lines!(ax1, t.s, t.beta_1, label=L"\beta_1")
    lines!(ax1, t.s, t.beta_2, label=L"\beta_2")
    axislegend(ax1, position=:rt)

    ax2 = Axis(f[2, 1], xlabel="s [m]", ylabel="Dispersion [m]")
    lines!(ax2, t.s, ηx, label=L"\eta_x")
    lines!(ax2, t.s, ηy, label=L"\eta_y")
    axislegend(ax2, position=:rt)

    return f
end

plot_linear_optics(tw0)

## 1.4. Build the merit function

(Need to add the data structure part)

We now turn the design goal into a scalar **merit function**.  
Optimization is formulated by minimizing a merit function of the form

$$
\mathcal{M} \equiv \sum_i w_i \,[\delta D_i]^2 + \sum_j w_j \,[\delta V_j]^2,
$$

where the first sum runs over the **data** and the second sum runs over the **variables**.  
The quantities $\delta D_i$ measure how far the current lattice is from the desired optics targets, while the weights $w_i$ determine how strongly each target is enforced.

For the present one-cell FODO example, the design goal is simple: at the **end of the cell**, the horizontal and vertical phase advances should both be $90^\circ$. In other words, we want

$$
\mu_x = \frac{\pi}{2}, \qquad \mu_y = \frac{\pi}{2}.
$$

A natural version of the merit function is therefore

$$
J(k_{QF}, k_{QD})
=
w_x \left(\mu_x - \frac{\pi}{2}\right)^2
+
w_y \left(\mu_y - \frac{\pi}{2}\right)^2,
$$

where $\mu_x$ and $\mu_y$ are the phase advances of the periodic one-cell optics, and $k_{QF}, k_{QD}$ are the quadrupole strengths to be varied.

We choose the optimization targets

- `phase.a = π/2`
- `phase.b = π/2`

and assign both targets the same weight,

$$
w_x = w_y = 10.
$$

With this choice, the merit function becomes

$$
J(k_{QF}, k_{QD})
=
10 \left(\mu_x - \frac{\pi}{2}\right)^2
+
10 \left(\mu_y - \frac{\pi}{2}\right)^2.
$$

In [ ]:
mu_target = π / 2
weight_x = 10.0
weight_y = 10.0

function merit(k)
    μx, μy = cell_phase_advances(k)
    return weight_x * (μx - mu_target)^2 + weight_y * (μy - mu_target)^2
end

println("Initial merit = ", merit([kqf0, kqd0]))

## 1.5. Differentiate the objective

`twiss` internally uses a six-variable GTPSA descriptor for the phase-space coordinates. In SciBmad 0.1.x, applying `AutoGTPSA()` directly to the two quadrupole strengths creates a separate two-variable descriptor, causing the error `Number of variables + parameters in GTPSAs do not agree!`.

For compatibility with the installed SciBmad version, we evaluate the gradient using a centered finite difference. The small two-variable problem is inexpensive, and the resulting gradient can still be supplied directly to `Optim.jl`.

In [ ]:
k0 = [kqf0, kqd0]

function merit_gradient!(storage, k)
    for i in eachindex(k)
        h = cbrt(eps(Float64)) * max(abs(k[i]), 1.0)
        kp = copy(k)
        km = copy(k)
        kp[i] += h
        km[i] -= h
        storage[i] = (merit(kp) - merit(km)) / (2h)
    end
    return storage
end

g0 = zeros(length(k0))
merit_gradient!(g0, k0)

println("Gradient of the merit at the initial point:")
println(g0)

## 1.6. Optimize the quadrupole strengths

We now pass both the objective and the user-supplied gradient to `Optim.jl`.


In [ ]:
g! = merit_gradient!

# Keep the search inside the linearly stable FODO region.
lower = [0.2, -0.6]
upper = [0.6, -0.2]

result = optimize(
    merit,
    g!,
    lower,
    upper,
    copy(k0),
    Fminbox(LBFGS()),
    Optim.Options(
        iterations=200,
        show_trace=true,
        show_every=1,
    ),
)

kopt = Optim.minimizer(result)

println()
println("Optimization converged? ", Optim.converged(result))
println("Optimized strengths:")
println("  KQF = ", kopt[1])
println("  KQD = ", kopt[2])
println("Final merit = ", Optim.minimum(result))

## 1.7. Verify the optimized optics

We finish with two checks:

1. verify numerically that the phase advances are now close to $90^\circ$;
2. compare the optics before and after optimization.


In [ ]:
tw_opt = twiss(build_fodo_cell(kopt[1], kopt[2]))

μx_opt, μy_opt = cell_phase_advances(kopt)

println("Optimized phase advances:")
println("  μx = ", μx_opt, " rad = ", μx_opt * 180 / π, " deg")
println("  μy = ", μy_opt, " rad = ", μy_opt * 180 / π, " deg")

In [ ]:
function compare_optics(tw_initial, tw_final)
    ti = optics_table(tw_initial)
    tf = optics_table(tw_final)

    ηxi, ηyi = linear_dispersion(ti)
    ηxf, ηyf = linear_dispersion(tf)

    f = Figure(fontsize=20, size=(1000, 900))

    ax1 = Axis(f[1, 1], xlabel="s [m]", ylabel="Beta [m]", title="Mode-1 beta")
    lines!(ax1, ti.s, ti.beta_1, linestyle=:dash, label="initial")
    lines!(ax1, tf.s, tf.beta_1, label="optimized")
    axislegend(ax1, position=:rt)

    ax2 = Axis(f[2, 1], xlabel="s [m]", ylabel="Beta [m]", title="Mode-2 beta")
    lines!(ax2, ti.s, ti.beta_2, linestyle=:dash, label="initial")
    lines!(ax2, tf.s, tf.beta_2, label="optimized")
    axislegend(ax2, position=:rt)

    ax3 = Axis(f[3, 1], xlabel="s [m]", ylabel="Dispersion [m]", title="Horizontal dispersion")
    lines!(ax3, ti.s, ηxi, linestyle=:dash, label="initial")
    lines!(ax3, tf.s, ηxf, label="optimized")
    axislegend(ax3, position=:rt)

    return f
end

compare_optics(tw0, tw_opt)

## 1.8. Exercises

The following exercises are adapted directly from the original Chapter 1 tutorial, but are phrased for the present SciBmad/Julia workflow.

1. **Reverse arc FODO.**  
   Construct the reverse arc FODO cell without sextupoles, and determine the quadrupole strengths that give a $90^\circ$ phase advance in both planes.

2. **Forward straight-section FODO.**  
   Replace the dipoles in the forward cell by a drift of length $5.855\ \mathrm{m}$ and rename these drifts `DB`. This length is chosen to produce the same straight-section length as in the RHIC tunnel. Rename the horizontally focusing and defocusing quadrupoles `QFSS` and `QDSS`, respectively. Which quadrupole strengths lead to a $90^\circ$ phase advance in both planes?

3. **Reverse straight-section FODO.**  
   Replace the dipoles in the reverse cell by a drift of length $5.855\ \mathrm{m}$ and rename these drifts `DB`. Rename the horizontally focusing and defocusing quadrupoles `QFSS` and `QDSS`, respectively. Which quadrupole strengths lead to a $90^\circ$ phase advance in both planes? The optimized quadrupole strengths should be exactly the same as in 1.8.3.

4. **Analytical FODO cell.**  
   Consider a FODO model with the same total length as constructed above, but with zero-length quadrupoles and no bends. Calculate by hand the quadrupole strengths that give a $90^\circ$ phase advance in both $x$ and $y$. What are the beta functions at the thin quadrupoles? How do the quadrupole strengths and beta functions compare with the numerical values found above?

5. **Forward and reverse cells.**  
   Check that the forward and reverse arc FODO cells, both starting with a focusing quadrupole, have different periodic beta and alpha functions. Check also that for the same phase advance of $90^\circ$, they have exactly the same quadrupole strengths. Explain why this is possible.

Still Need to explore how to transfer exercises 6-8.

